In [1]:
from utils import createSparkSession, getThroughputRate, read_csv_from_s3, write_postgres_table
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, ByteType, ShortType

In [2]:
spark, stagemetrics = createSparkSession('P1M1')

26/11/05 05:16:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


## M0-2 Stage as text

In [3]:
stagemetrics.begin()

In [4]:
aisles_schema = StructType([
    StructField('aisle_id', StringType(), False),
    StructField('aisle', StringType(), True),
    StructField('bad_record', StringType(), True),
])

departments_schema = StructType([
    StructField('department_id', StringType(), False),
    StructField('department', StringType(), True),
    StructField('bad_record', StringType(), True),
])

order_products_schema = StructType([
    StructField('product_id', StringType(), False),
    StructField('order_id', StringType(), False),
    StructField('add_to_cart_order', StringType(), True),
    StructField('reordered', StringType(), True),
    StructField('bad_record', StringType(), True),
])

products_schema = StructType([
    StructField('product_id', StringType(), False),
    StructField('product_name', StringType(), True),
    StructField('aisle_id', StringType(), True),
    StructField('department_id', StringType(), True),
    StructField('bad_record', StringType(), True),
])

orders_schema = StructType([
    StructField('order_id', StringType(), False),
    StructField('user_id', StringType(), False),
    StructField('eval_set', StringType(), True),
    StructField('order_number', StringType(), True),
    StructField('order_dow', StringType(), True),
    StructField('order_hour_of_day', StringType(), True),
    StructField('days_since_prior_order', StringType(), True),
    StructField('bad_record', StringType(), True),
])

In [5]:
aisles_df = read_csv_from_s3(spark, "s3a://raw/InstaCart/aisles.csv", schema=aisles_schema, options={'escape':'"'})
departments_df = read_csv_from_s3(spark, "s3a://raw/InstaCart/departments.csv", schema=departments_schema, options={'escape':'"'})
order_products__prior_df = read_csv_from_s3(spark, "s3a://raw/InstaCart/order_products__prior.csv", schema=order_products_schema, options={'escape':'"'})
order_products__train_df = read_csv_from_s3(spark, "s3a://raw/InstaCart/order_products__train.csv", schema=order_products_schema, options={'escape':'"'})
orders_df = read_csv_from_s3(spark, "s3a://raw/InstaCart/orders.csv", schema=orders_schema, options={'escape':'"'})
products_df = read_csv_from_s3(spark, "s3a://raw/InstaCart/products.csv", schema=products_schema, options={'escape':'"'})

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


In [6]:
products_df.filter('bad_record is not null').show()

[Stage 0:>                                                          (0 + 1) / 1]

+----------+------------+--------+-------------+----------+
|product_id|product_name|aisle_id|department_id|bad_record|
+----------+------------+--------+-------------+----------+
+----------+------------+--------+-------------+----------+



In [7]:
products_df.filter('product_id = 6816').show()

[Stage 1:>                                                          (0 + 1) / 1]

+----------+--------------------+--------+-------------+----------+
|product_id|        product_name|aisle_id|department_id|bad_record|
+----------+--------------------+--------+-------------+----------+
|      6816|Scotch Kids 5\" S...|      87|           17|      NULL|
+----------+--------------------+--------+-------------+----------+



In [8]:
order_products_df = order_products__prior_df.union(order_products__train_df)

In [9]:
write_postgres_table(aisles_df, 'staging_text.aisles')
write_postgres_table(departments_df, 'staging_text.departments')
write_postgres_table(order_products_df, 'staging_text.order_products')
write_postgres_table(orders_df, 'staging_text.orders')
write_postgres_table(products_df, 'staging_text.products')

In [10]:
stagemetrics.end()

In [11]:
stagemetrics.print_report()
stagemetrics.print_memory_report()
getThroughputRate(stagemetrics.aggregate_stagemetrics())


Scheduling mode = FIFO
Spark Context default degree of parallelism = 9

Aggregated Spark stage metrics:
numStages => 7
numTasks => 29
elapsedTime => 130722 (2.2 min)
stageDuration => 129807 (2.2 min)
executorRunTime => 1074982 (18 min)
executorCpuTime => 129266 (2.2 min)
executorDeserializeTime => 4311 (4 s)
executorDeserializeCpuTime => 2255 (2 s)
resultSerializationTime => 35 (35 ms)
jvmGCTime => 41846 (42 s)
shuffleFetchWaitTime => 0 (0 ms)
shuffleWriteTime => 0 (0 ms)
resultSize => 26812 (26.2 KB)
diskBytesSpilled => 0 (0 Bytes)
memoryBytesSpilled => 0 (0 Bytes)
peakExecutionMemory => 0
recordsRead => 37389408
bytesRead => 719079486 (685.8 MB)
recordsWritten => 37290032
bytesWritten => 0 (0 Bytes)
shuffleRecordsRead => 0
shuffleTotalBlocksFetched => 0
shuffleLocalBlocksFetched => 0
shuffleRemoteBlocksFetched => 0
shuffleTotalBytesRead => 0 (0 Bytes)
shuffleLocalBytesRead => 0 (0 Bytes)
shuffleRemoteBytesRead => 0 (0 Bytes)
shuffleRemoteBytesReadToDisk => 0 (0 Bytes)
shuffleBytesWr

26/11/05 05:18:28 WARN StageMetrics: No executor metrics found for stage 6 and metric JVMHeapMemory.
26/11/05 05:18:28 WARN StageMetrics: No executor metrics found for stage 6 and metric OnHeapExecutionMemory.


## M0-3 Parsing fixes

In [12]:
stagemetrics.begin()

In [13]:
aisles_schema = StructType([
    StructField('aisle_id', IntegerType(), False),
    StructField('aisle', StringType(), True),
    StructField('bad_record', StringType(), True),
])

departments_schema = StructType([
    StructField('department_id', IntegerType(), False),
    StructField('department', StringType(), True),
    StructField('bad_record', StringType(), True),
])

order_products_schema = StructType([
    StructField('product_id', IntegerType(), False),
    StructField('order_id', IntegerType(), False),
    StructField('add_to_cart_order', IntegerType(), True),
    StructField('reordered', ByteType(), True),
    StructField('bad_record', StringType(), True),
])

products_schema = StructType([
    StructField('product_id', IntegerType(), False),
    StructField('product_name', StringType(), True),
    StructField('aisle_id', IntegerType(), True),
    StructField('department_id', IntegerType(), True),
    StructField('bad_record', StringType(), True),
])

orders_schema = StructType([
    StructField('order_id', IntegerType(), False),
    StructField('user_id', IntegerType(), False),
    StructField('eval_set', StringType(), True),
    StructField('order_number', IntegerType(), True),
    StructField('order_dow', IntegerType(), True),
    StructField('order_hour_of_day', IntegerType(), True),
    StructField('days_since_prior_order', DoubleType(), True),
    StructField('bad_record', StringType(), True),
])

In [14]:
aisles_df = read_csv_from_s3(spark, "s3a://raw/InstaCart/aisles.csv", schema=aisles_schema, options={'escape':'"'})
departments_df = read_csv_from_s3(spark, "s3a://raw/InstaCart/departments.csv", schema=departments_schema, options={'escape':'"'})
order_products__prior_df = read_csv_from_s3(spark, "s3a://raw/InstaCart/order_products__prior.csv", schema=order_products_schema, options={'escape':'"'})
order_products__train_df = read_csv_from_s3(spark, "s3a://raw/InstaCart/order_products__train.csv", schema=order_products_schema, options={'escape':'"'})
orders_df = read_csv_from_s3(spark, "s3a://raw/InstaCart/orders.csv", schema=orders_schema, options={'escape':'"'})
products_df = read_csv_from_s3(spark, "s3a://raw/InstaCart/products.csv", schema=products_schema, options={'escape':'"'})

products csv has a bad record; issue is csv is standard csv with double quote as escape character

In [15]:
products_df.filter('bad_record is not NULL').show()

+----------+------------+--------+-------------+----------+
|product_id|product_name|aisle_id|department_id|bad_record|
+----------+------------+--------+-------------+----------+
+----------+------------+--------+-------------+----------+



In [17]:
products_df.filter('product_id = 6816').show()

+----------+--------------------+--------+-------------+----------+
|product_id|        product_name|aisle_id|department_id|bad_record|
+----------+--------------------+--------+-------------+----------+
|      6816|Scotch Kids 5\" S...|      87|           17|      NULL|
+----------+--------------------+--------+-------------+----------+



In [18]:
order_products_df = order_products__prior_df.union(order_products__train_df)

In [19]:
write_postgres_table(aisles_df, 'staging.aisles')
write_postgres_table(departments_df, 'staging.departments')
write_postgres_table(order_products_df, 'staging.order_products')
write_postgres_table(orders_df, 'staging.orders')
write_postgres_table(products_df, 'staging.products')

In [20]:
stagemetrics.end()

In [21]:
stagemetrics.print_report()
stagemetrics.print_memory_report()
getThroughputRate(stagemetrics.aggregate_stagemetrics())


Scheduling mode = FIFO
Spark Context default degree of parallelism = 9

Aggregated Spark stage metrics:
numStages => 7
numTasks => 29
elapsedTime => 117456 (2.0 min)
stageDuration => 110128 (1.8 min)
executorRunTime => 960233 (16 min)
executorCpuTime => 101309 (1.7 min)
executorDeserializeTime => 652 (0.7 s)
executorDeserializeCpuTime => 426 (0.4 s)
resultSerializationTime => 48 (48 ms)
jvmGCTime => 35507 (36 s)
shuffleFetchWaitTime => 0 (0 ms)
shuffleWriteTime => 0 (0 ms)
resultSize => 26941 (26.3 KB)
diskBytesSpilled => 0 (0 Bytes)
memoryBytesSpilled => 0 (0 Bytes)
peakExecutionMemory => 0
recordsRead => 37339721
bytesRead => 719079486 (685.8 MB)
recordsWritten => 37290032
bytesWritten => 0 (0 Bytes)
shuffleRecordsRead => 0
shuffleTotalBlocksFetched => 0
shuffleLocalBlocksFetched => 0
shuffleRemoteBlocksFetched => 0
shuffleTotalBytesRead => 0 (0 Bytes)
shuffleLocalBytesRead => 0 (0 Bytes)
shuffleRemoteBytesRead => 0 (0 Bytes)
shuffleRemoteBytesReadToDisk => 0 (0 Bytes)
shuffleBytesW

26/11/05 05:20:40 WARN StageMetrics: No executor metrics found for stage 13 and metric JVMHeapMemory.
26/11/05 05:20:40 WARN StageMetrics: No executor metrics found for stage 13 and metric OnHeapExecutionMemory.


## M0-5 Profiling

In [22]:
aisles_df.createOrReplaceTempView('aisles')
departments_df.createOrReplaceTempView('departments')
order_products_df.createOrReplaceTempView('order_products')
orders_df.createOrReplaceTempView('orders')
products_df.createOrReplaceTempView('products')

In [65]:
spark.sql('''
select *
from  orders 
where orders.eval_set='test' 
''').show()

+--------+-------+--------+------------+---------+-----------------+----------------------+----------+
|order_id|user_id|eval_set|order_number|order_dow|order_hour_of_day|days_since_prior_order|bad_record|
+--------+-------+--------+------------+---------+-----------------+----------------------+----------+
| 2774568|      3|    test|          13|        5|               15|                  11.0|      NULL|
|  329954|      4|    test|           6|        3|               12|                  30.0|      NULL|
| 1528013|      6|    test|           4|        3|               16|                  22.0|      NULL|
| 1376945|     11|    test|           8|        6|               11|                   8.0|      NULL|
| 1356845|     12|    test|           6|        1|               20|                  30.0|      NULL|
| 2161313|     15|    test|          23|        1|                9|                   7.0|      NULL|
| 1416320|     16|    test|           7|        0|               13|     

In [71]:
spark.sql('''
select *
from order_products
where order_id = 2774568
''').show()

[Stage 134:=================================>                      (6 + 4) / 10]

+----------+--------+-----------------+---------+----------+
|product_id|order_id|add_to_cart_order|reordered|bad_record|
+----------+--------+-----------------+---------+----------+
+----------+--------+-----------------+---------+----------+



In [67]:
spark.sql('''
select *
from  orders 
where orders.eval_set='train' 
''').show()

+--------+-------+--------+------------+---------+-----------------+----------------------+----------+
|order_id|user_id|eval_set|order_number|order_dow|order_hour_of_day|days_since_prior_order|bad_record|
+--------+-------+--------+------------+---------+-----------------+----------------------+----------+
| 1187899|      1|   train|          11|        4|                8|                  14.0|      NULL|
| 1492625|      2|   train|          15|        1|               11|                  30.0|      NULL|
| 2196797|      5|   train|           5|        0|               11|                   6.0|      NULL|
|  525192|      7|   train|          21|        2|               11|                   6.0|      NULL|
|  880375|      8|   train|           4|        1|               14|                  10.0|      NULL|
| 1094988|      9|   train|           4|        6|               10|                  30.0|      NULL|
| 1822501|     10|   train|           6|        0|               19|     

In [68]:
spark.sql('''
select *
from order_products
where order_id = 1187899
''').show()

[Stage 127:=================================>                      (6 + 4) / 10]

+----------+--------+-----------------+---------+----------+
|product_id|order_id|add_to_cart_order|reordered|bad_record|
+----------+--------+-----------------+---------+----------+
+----------+--------+-----------------+---------+----------+



In [69]:
spark.sql('''
select *
from  orders 
where orders.eval_set='prior' 
''').show()

+--------+-------+--------+------------+---------+-----------------+----------------------+----------+
|order_id|user_id|eval_set|order_number|order_dow|order_hour_of_day|days_since_prior_order|bad_record|
+--------+-------+--------+------------+---------+-----------------+----------------------+----------+
| 2539329|      1|   prior|           1|        2|                8|                  NULL|      NULL|
| 2398795|      1|   prior|           2|        3|                7|                  15.0|      NULL|
|  473747|      1|   prior|           3|        3|               12|                  21.0|      NULL|
| 2254736|      1|   prior|           4|        4|                7|                  29.0|      NULL|
|  431534|      1|   prior|           5|        4|               15|                  28.0|      NULL|
| 3367565|      1|   prior|           6|        2|                7|                  19.0|      NULL|
|  550135|      1|   prior|           7|        1|                9|     

In [70]:
spark.sql('''
select *
from order_products
where order_id = 2398795
''').show()

[Stage 131:=================================>                      (6 + 4) / 10]

+----------+--------+-----------------+---------+----------+
|product_id|order_id|add_to_cart_order|reordered|bad_record|
+----------+--------+-----------------+---------+----------+
+----------+--------+-----------------+---------+----------+



In [8]:
from data_profiling import ProfileReport

In [9]:
ProfileReport(aisles_df).to_file('/home/ubuntu/work_dir/profiling/aisles.html')

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

26/10/31 11:14:38 WARN CacheManager: Asked to cache already cached data.        
26/10/31 11:14:41 WARN CacheManager: Asked to cache already cached data.


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [10]:
ProfileReport(departments_df).to_file('/home/ubuntu/work_dir/profiling/departments.html')

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

26/10/31 11:14:53 WARN CacheManager: Asked to cache already cached data.
26/10/31 11:14:54 WARN CacheManager: Asked to cache already cached data.


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [11]:
ProfileReport(order_products__prior_df).to_file('/home/ubuntu/work_dir/profiling/order_products__prior.html')

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

26/10/31 11:16:23 WARN CacheManager: Asked to cache already cached data.        
26/10/31 11:16:45 WARN CacheManager: Asked to cache already cached data.        
26/10/31 11:17:04 WARN CacheManager: Asked to cache already cached data.        
26/10/31 11:17:17 WARN CacheManager: Asked to cache already cached data.        
                                                                                

Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [12]:
ProfileReport(order_products__train_df).to_file('/home/ubuntu/work_dir/profiling/order_products__train.html')

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

26/10/31 11:18:37 WARN CacheManager: Asked to cache already cached data.
26/10/31 11:18:41 WARN CacheManager: Asked to cache already cached data.        
26/10/31 11:18:45 WARN CacheManager: Asked to cache already cached data.        
26/10/31 11:18:47 WARN CacheManager: Asked to cache already cached data.
                                                                                

Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [13]:
ProfileReport(orders_df).to_file('/home/ubuntu/work_dir/profiling/orders.html')

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

26/10/31 11:19:05 WARN CacheManager: Asked to cache already cached data.        
26/10/31 11:19:15 WARN CacheManager: Asked to cache already cached data.        
26/10/31 11:19:20 WARN CacheManager: Asked to cache already cached data.        
26/10/31 11:19:23 WARN CacheManager: Asked to cache already cached data.        
26/10/31 11:19:27 WARN CacheManager: Asked to cache already cached data.        
26/10/31 11:19:30 WARN CacheManager: Asked to cache already cached data.        
26/10/31 11:19:34 WARN CacheManager: Asked to cache already cached data.        
                                                                                

Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [14]:
ProfileReport(products_df).to_file('/home/ubuntu/work_dir/profiling/products.html')

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

26/10/31 11:19:49 WARN CacheManager: Asked to cache already cached data.
26/10/31 11:19:50 WARN CacheManager: Asked to cache already cached data.
26/10/31 11:19:52 WARN CacheManager: Asked to cache already cached data.
26/10/31 11:19:54 WARN CacheManager: Asked to cache already cached data.


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [13]:
spark.stop()